In [49]:
import requests

class OpenRouter:
    def __init__(self, api_key: str, model: str = "meta-llama/llama-3.2-3b-instruct", title="openrouter-call"):
        self.api_key = api_key
        self.model = model
        self.url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "X-Title": title,
            "Content-Type": "application/json"
        }

    def chat(self, prompt: str, temperature=0.3, max_tokens=500):
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        response = requests.post(self.url, headers=self.headers, json=payload)

        if response.status_code != 200:
            raise Exception(f"OpenRouter Error {response.status_code}: {response.text}")
        
        return response.json()['choices'][0]['message']['content'].strip()


In [50]:
import random
import json
from typing import List, Tuple

class PromptBuilder:
    def __init__(self, train_data_path: str, mode: str = 'zero-shot', shot_num: int = 0):
        """
        :param train_data_path: path to generation_train.jsonl
        :param mode: one of ['zero-shot', 'few-shot', 'many-shot']
        :param shot_num: number of examples to include (ignored for zero-shot)
        """
        self.mode = mode
        self.shot_num = shot_num

        with open(train_data_path, 'r', encoding='utf-8') as f:
            self.examples = [json.loads(line.strip()) for line in f]

    def build_prompt(self, input_question: str) -> str:
        """Return prompt string ready for LLM input"""
        instruction = "Translate the following natural language question into an SQL query."
        prompt = instruction + "\n\n"

        if self.mode == 'zero-shot':
            prompt += f"Question: {input_question}\nSQL:"
        else:
            samples = random.sample(self.examples, min(self.shot_num, len(self.examples)))
            for ex in samples:
                prompt += f"Question: {ex['text']}\nSQL: {ex['sql']}\n\n"
            prompt += f"Question: {input_question}\nSQL:"

        return prompt


# Example usage
if __name__ == '__main__':
    builder = PromptBuilder('../../processed_data/question_split/generation_train.jsonl', mode='zero-shot')
    q = "what are the flights from dallas to boston on july 5"
    prompt = builder.build_prompt(q)
    print(prompt)


Translate the following natural language question into an SQL query.

Question: what are the flights from dallas to boston on july 5
SQL:


In [51]:
import json
import random
import re
import csv
from collections import defaultdict
from tqdm import tqdm
# from openrouter_sdk import OpenRouter

# ========== 配置 ==========
API_KEY =  "YOUR_OPENROUTER_API_KEY"
MODEL = "meta-llama/llama-3.2-3b-instruct"
client = OpenRouter(api_key=API_KEY, model=MODEL)

# ========== 工具函数 ==========
def clean_sql(sql):
    return re.sub(r"\s+", " ", sql.strip().lower())

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line.strip()) for line in f if line.strip()]

def evaluate(pred, refs):
    pred = clean_sql(pred)
    refs = [clean_sql(r) for r in refs]
    return pred in refs

# ========== 主评估函数 ==========
def run_eval(train_file, test_file, shot_type="zero-shot", shot_count=0, n=5):
    builder = PromptBuilder(train_file, mode=shot_type, shot_num=shot_count)
    test_data = load_jsonl(test_file)[:n]

    correct = 0
    total = len(test_data)

    for idx, item in enumerate(tqdm(test_data)):
        prompt = builder.build_prompt(item['text'])
        try:
            gen_sql = client.chat(prompt)
        except Exception as e:
            print(f"❌ Error on item {idx}: {e}")
            continue

        if evaluate(gen_sql, item['sql']):
            correct += 1

        print(f"🧪 Q: {item['text']}, \n 🔧 PRED: {gen_sql}, \n🎯 GOLD: {item['sql']}")

    acc = correct / total if total > 0 else 0
    print(f"✅ [{shot_type} | {shot_count} shots] Accuracy: {acc:.4f} ({correct}/{total})")


In [54]:
# from task3_openrouter import run_eval
import json

# 手动加载前 5 条样本写入一个新文件
def write_subset(file_path, out_path, n=5):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line for _, line in zip(range(n), f) if line.strip()]
    with open(out_path, "w", encoding="utf-8") as out:
        out.writelines(lines)

write_subset("generation_test.jsonl", "generation_test_5.jsonl", n=5)

# 然后运行评估（比如 few-shot 5）
run_eval(
    train_file="generation_train.jsonl",
    test_file="generation_test_5.jsonl",  # 子集
    # schema_csv_file="atis-schema.csv",
    shot_type="few-shot",
    shot_count=5,
    n=5  # 控制评估数量
)


  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:08<00:34,  8.72s/it]

🧪 Q: do you have an 819 flight from DENVER to SAN FRANCISCO, 
 🔧 PRED: Here are the SQL queries translated from the given natural language questions:

**1. Show me the flights from NEWARK to LOS ANGELES**
```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE, AIRPORT_SERVICE, CITY, CITY, FLIGHT
WHERE CITY.CITY_CODE = AIRPORT_SERVICE.CITY_CODE
AND CITY.CITY_NAME = 'NEWARK'
AND CITY.CITY_CODE = AIRPORT_SERVICE2.CITY_CODE
AND CITY.CITY_NAME = 'LOS ANGELES'
AND FLIGHT.FROM_AIRPORT = AIRPORT_SERVICE.CITY_CODE
AND FLIGHT.TO_AIRPORT = AIRPORT_SERVICE2.AIRPORT_CODE;
```
**2. Information on a flight from SAN FRANCISCO to PHILADELPHIA**
```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE, AIRPORT_SERVICE, CITY, CITY, FLIGHT
WHERE CITY.CITY_CODE = AIRPORT_SERVICE.CITY_CODE
AND CITY.CITY_NAME = 'SAN FRANCISCO'
AND CITY.CITY_CODE = AIRPORT_SERVICE2.CITY_CODE
AND CITY.CITY_NAME = 'PHILADELPHIA'
AND FLIGHT.FROM_AIRPORT = AIRPORT_SERVICE.CITY_CODE
AND FLIGHT.TO_AIRPORT = AIRPORT_SERVICE2.AIRPORT_COD

 40%|████      | 2/5 [00:11<00:16,  5.47s/it]

🧪 Q: i would like a flight from KANSAS CITY to ST. PAUL departing at 1600, 
 🔧 PRED: Here are the translations of the given natural language questions into SQL queries:

**1. Nonstop flights from NASHVILLE to ST. LOUIS**

```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE, AIRPORT_SERVICE, CITY, CITY, FLIGHT
WHERE CITY.CITY_CODE = AIRPORT_SERVICE.CITY_CODE
AND CITY.CITY_NAME = 'NASHVILLE'
AND AIRPORT_SERVICE2.CITY_CODE = AIRPORT_SERVICE2.AIRPORT_CODE
AND AIRPORT_SERVICE2.CITY_NAME = 'ST. LOUIS'
AND FLIGHT.FLIGHT_ID = AIRPORT_SERVICE2.AIRPORT_CODE
AND FLIGHT.STOPS = 0;
```

**2. Flights from BOSTON to OAKLAND stopping in DENVER**

```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE, AIRPORT_SERVICE, AIRPORT_SERVICE, CITY, CITY, CITY, FLIGHT, FLIGHT_STOP
WHERE CITY.CITY_CODE = AIRPORT_SERVICE.CITY_CODE
AND CITY.CITY_NAME = 'OAKLAND'
AND AIRPORT_SERVICE2.CITY_CODE = AIRPORT_SERVICE2.AIRPORT_CODE
AND AIRPORT_SERVICE3.CITY_CODE = AIRPORT_SERVICE3.AIRPORT_CODE
AND CITY.CITY_NAME = 'DENV

...

🧪 Q: do you have a flight leaving BOSTON at 645 going to WASHINGTON, 
 🔧 PRED: Here are the translations of the natural language questions into SQL queries:

**Question 1: Show me all flights from SAN FRANCISCO to OAKLAND**
```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE,
     CITY,
     FLIGHT
WHERE CITY_CODE = AIRPORT_SERVICE.CITY_CODE
AND CITY_NAME = 'SAN FRANCISCO'
AND CITY_CODE = AIRPORT_SERVICE2.CITY_CODE
AND CITY_NAME = 'OAKLAND'
AND FROM_AIRPORT = AIRPORT_SERVICE.AIRPORT_CODE
AND TO_AIRPORT = AIRPORT_SERVICE2.AIRPORT_CODE;
```
**Question 2: Is there ground transportation in DFW from the DFW airport to downtown DALLAS**
```sql
SELECT DISTINCT TRANSPORT_TYPE
FROM GROUND_SERVICE,
     AIRPORT,
     CITY
WHERE AIRPORT_CODE = 'DFW'
AND CITY_NAME = 'DALLAS'
AND AIRPORT_CODE = GROUND_SERVICE.AIRPORT_CODE
AND CITY_CODE = CITY.CITY_CODE;
```
**Question 3: Show me the nonstop flights from DALLAS to HOUSTON**
```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE,
     CITY,
     FLI

 80%|████████  | 4/5 [00:21<00:04,  4.99s/it]

🧪 Q: i need a flight from PITTSBURGH to NEW YORK leaving at 1700, 
 🔧 PRED: Here are the SQL queries for each of the given natural language questions:

**Question 1: What flight do you have from SAN FRANCISCO to PITTSBURGH on the evening of 8/27?**

```sql
SELECT DISTINCT FLIGHTalias0.FLIGHT_ID 
FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0, 
     AIRPORT_SERVICE AS AIRPORT_SERVICEalias1, 
     CITY AS CITYalias0, 
     CITY AS CITYalias1, 
     DATE_DAY AS DATE_DAYalias0, 
     DAYS AS DAYSalias0, 
     FLIGHT AS FLIGHTalias0 
WHERE ( ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE 
         AND CITYalias1.CITY_NAME = "PITTSBURGH" 
         AND DATE_DAYalias0.DAY_NUMBER = 27 
         AND DATE_DAYalias0.MONTH_NUMBER = 8 
         AND DATE_DAYalias0.YEAR = 1991 
         AND DAYSalias0.DAY_NAME = DATE_DAYalias0.DAY_NAME 
         AND FLIGHTalias0.FLIGHT_DAYS = DAYSalias0.DAYS_CODE 
         AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE ) 
      AND CITYalias

100%|██████████| 5/5 [00:24<00:00,  4.95s/it]

🧪 Q: is there a 400 o'clock flight from WASHINGTON to DENVER, 
 🔧 PRED: Here are the translations:

**Question 1:**
i'd like to fly from PHILADELPHIA to DALLAS to SAN FRANCISCO

SQL:
```sql
SELECT DISTINCT FLIGHTalias0.FLIGHT_ID
FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0,
     AIRPORT_SERVICE AS AIRPORT_SERVICEalias1,
     AIRPORT_SERVICE AS AIRPORT_SERVICEalias2,
     CITY AS CITYalias0,
     CITY AS CITYalias1,
     CITY AS CITYalias2,
     FLIGHT AS FLIGHTalias0
WHERE ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE
        AND CITYalias1.CITY_NAME = 'SAN FRANCISCO'
        AND CITYalias2.CITY_CODE = AIRPORT_SERVICEalias2.CITY_CODE
        AND CITYalias2.CITY_NAME = 'DALLAS'
        AND FLIGHTalias0.FLIGHT_ID = FLIGHT_STOPalias0.FLIGHT_ID
        AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE )
  AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE
  AND CITYalias0.CITY_NAME = 'PHILADELPHIA'
  AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.A

In [32]:
print(prompt)
print(gen_sql)
print(item['sql'])

NameError: name 'prompt' is not defined